# Bayesian PMF Demo

Posterior uncertainty quantification with `pmf_bayes`: credible intervals on
recovered profiles and convergence diagnostics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from pmf_acls import pmf_bayes

rng = np.random.default_rng(42)

n_obs, n_vars, p = 80, 10, 4

# True profiles with varying magnitudes
G_true = rng.gamma(2, 1, size=(p, n_vars))
G_true[2:] *= 0.3  # minor factors
F_true = rng.gamma(2, 1, size=(n_obs, p))
X_true = F_true @ G_true

sigma = 0.15 * X_true + 0.01
X_obs = np.maximum(X_true + rng.normal(0, sigma), 0)

In [ ]:
result = pmf_bayes(
    X_obs, sigma, p=4,
    n_samples=800, n_burnin=400, n_thin=2,
    store_samples=True, random_seed=0,
)

print(f"Converged: {result.converged}")
print(f"Chi-squared: {result.chi2:.3f}")
print(f"Explained variance: {result.explained_variance:.4f}")
print(f"Posterior samples shape: {result.G_samples.shape}")

In [ ]:
# Normalize and align
def row_norm(M):
    return M / np.maximum(M.sum(axis=1, keepdims=True), 1e-30)

G_true_n = row_norm(G_true)
G_hat_n = row_norm(result.G)
G_samples_n = result.G_samples / np.maximum(
    result.G_samples.sum(axis=2, keepdims=True), 1e-30)

cos_mat = G_true_n @ G_hat_n.T / (
    np.linalg.norm(G_true_n, axis=1, keepdims=True) @
    np.linalg.norm(G_hat_n, axis=1, keepdims=True).T
)
row_ind, col_ind = linear_sum_assignment(1 - cos_mat)

# Profiles with 95% credible intervals
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
x = np.arange(n_vars)
for i, ax in enumerate(axes.ravel()):
    ci_lo = np.percentile(G_samples_n[:, col_ind[i]], 2.5, axis=0)
    ci_hi = np.percentile(G_samples_n[:, col_ind[i]], 97.5, axis=0)
    y = G_hat_n[col_ind[i]]
    yerr = np.vstack([np.maximum(y - ci_lo, 0), np.maximum(ci_hi - y, 0)])

    ax.bar(x - 0.15, G_true_n[row_ind[i]], 0.3, alpha=0.7, label="True")
    ax.bar(x + 0.15, y, 0.3, facecolor="none", edgecolor="C1", linewidth=1.5,
           yerr=yerr, ecolor="black", capsize=3, label="Bayesian (95% CI)")
    cs = cos_mat[row_ind[i], col_ind[i]]
    ax.set_title(f"Factor {i+1} (cos={cs:.3f})")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

fig.suptitle("Bayesian PMF: profiles with posterior credible intervals")
fig.tight_layout()
plt.show()